In [ ]:
# ==============================================================================
# PARALLEL DIM: MANAGER + STAFF
# ==============================================================================
from notebooks.helpers import (
    IncrementalPipeline, TableConfig, get_latest_batch_id, setup_logger,
    write_gold_table, safe_count, generate_batch_id,
)
from notebooks.helpers.silver_transforms import (
    transform_staff_full_pipeline,
    transform_manager_full_pipeline,
    build_staff_hierarchy_bridge,
)
import pandas as pd

logger = setup_logger("parallel_dim_manager_staff")

batch_id = generate_batch_id()
pipeline = IncrementalPipeline(spark, dbutils, batch_id=batch_id)

staff_batch_id = get_latest_batch_id(spark, "staff")
if not staff_batch_id:
    raise ValueError("No bronze batch_id found for staff; run bronze load first.")
logger.info(f"Using bronze batch_id for staff: {staff_batch_id}")

staff_config = TableConfig(
    table_name="staff",
    business_key="staff_id",
    surrogate_key="staff_key",
    watermark_column="last_update",
    scd_type=1,
    gold_table_name="dim_staff",
    silver_transform=transform_staff_full_pipeline,
    dependencies=["address", "city", "country"],
)

manager_config = TableConfig(
    table_name="staff",
    business_key="staff_id",
    surrogate_key="staff_key",
    watermark_column="last_update",
    scd_type=1,
    gold_table_name="dim_manager",
    silver_transform=transform_manager_full_pipeline,
    dependencies=["address", "city", "country"],
)

print("Row Counts (Before):")
print(f"dim_staff: {safe_count(spark, 'dim_staff')}")
print(f"dim_manager: {safe_count(spark, 'dim_manager')}")
print(f"bridge_staff_hierarchy: {safe_count(spark, 'bridge_staff_hierarchy')}")
print("\nLatest Watermarks (Before):")
display(spark.table("wheelie.monitoring.watermarks"))


In [ ]:
results = pipeline.load_tables(
    [staff_config, manager_config],
    force_full=False,
    bronze_batch_id=staff_batch_id,
)

# Rebuild staff hierarchy bridge after staff load
staff_bronze = spark.table("wheelie.bronze.staff")
bridge_staff_hierarchy = build_staff_hierarchy_bridge(staff_bronze, max_depth=10)
write_gold_table(bridge_staff_hierarchy, "bridge_staff_hierarchy", mode="overwrite")

display(pd.DataFrame(results))

print("\nRow Counts (After):")
print(f"dim_staff: {safe_count(spark, 'dim_staff')}")
print(f"dim_manager: {safe_count(spark, 'dim_manager')}")
print(f"bridge_staff_hierarchy: {safe_count(spark, 'bridge_staff_hierarchy')}")
print("\nLatest Watermarks (After):")
display(spark.table("wheelie.monitoring.watermarks"))
